# recipe-dataclass — faded example 3: Recipe kwargs for clamp_min_forward

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `recipe-dataclass`. Running the beacon reports progress on the `Backprop: Recipe dataclass` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Recipe dataclass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`recipe-dataclass`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "recipe-dataclass"
DD_SUBTOPIC = "Backprop: Recipe dataclass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Non-Tensor configuration values (like a `min` threshold) go in `kwargs`, keyed exactly as the backward function expects, while only Tensor inputs go in `parents`. The kwargs are splatted into the backward call, so the key must be `'min'`, not the local parameter name.

## Faded exercise 3

Complete `clamp_min_forward(x, *, min_val)` for a `MiniTensor`, computing `out = x.array.clamp(min=min_val)`. The `func`, `args`, and `parents` are filled. Fill in the `kwargs` dict so it carries the threshold under the exact key `'min'`.

**Fill in:** build the kwargs dict carrying the min threshold under the key 'min'

In [ ]:
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def clamp_min_forward(x: MiniTensor, *, min_val: float) -> MiniTensor:
    out_arr = x.array.clamp(min=min_val)
    kwargs = None  # TODO: build the kwargs dict carrying the min threshold under the key 'min'
    recipe = Recipe(
        func=t.clamp,
        args=(x.array,),
        kwargs=kwargs,
        parents={0: x},
    )
    return MiniTensor(out_arr, recipe=recipe)


x = MiniTensor(t.tensor([-1.0, 0.5, 2.0]))
out = clamp_min_forward(x, min_val=0.0)

def _test():
    x = MiniTensor(t.tensor([-1.0, 0.5, 2.0]))
    out = clamp_min_forward(x, min_val=0.0)
    r = out.recipe
    assert r.kwargs == {'min': 0.0}, "kwargs must be {'min': 0.0} with exact key 'min'"
    assert set(r.parents.keys()) == {0}, 'parents must key only argnum 0 (min is not a parent)'
    assert t.allclose(out.array, t.tensor([0.0, 0.5, 2.0])), 'forward must clamp below at 0'

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def clamp_min_forward(x: MiniTensor, *, min_val: float) -> MiniTensor:
    out_arr = x.array.clamp(min=min_val)
    kwargs = {'min': min_val}
    recipe = Recipe(
        func=t.clamp,
        args=(x.array,),
        kwargs=kwargs,
        parents={0: x},
    )
    return MiniTensor(out_arr, recipe=recipe)


x = MiniTensor(t.tensor([-1.0, 0.5, 2.0]))
out = clamp_min_forward(x, min_val=0.0)
```
</details>